In [1]:
import torch
from torchviz import make_dot

x = torch.tensor(2.0, requires_grad=True)
y = x * x + 3 * x + 1

dot = make_dot(y, params={"x": x})
dot.render("simple_graph", format="png")


'simple_graph.png'

In [57]:
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont


def tensor_weights_to_image(
    tensor: torch.Tensor,
    out_path: str = None,
    cell_size: int = 1,
    header_height: int = 0,
    margin: int = 0,
    scale: float = None,
    background=(0, 0, 0),
    text_color=(240, 240, 240),
):
    if not isinstance(tensor, torch.Tensor):
        raise TypeError("tensor must be a torch.Tensor")

    t = tensor.detach().float().cpu()

    if t.numel() == 0:
        raise ValueError("tensor must not be empty")

    # Make tensor 2D for visualization
    if t.ndim == 0:
        t = t.reshape(1, 1)
    elif t.ndim == 1:
        t = t.unsqueeze(0)
    elif t.ndim > 2:
        t = t.reshape(-1, t.shape[-1])

    h, w = t.shape
    arr = t.numpy()

    mean_val = float(arr.mean())
    var_val = float(arr.var())

    # Symmetric scaling around zero
    if scale is None:
        scale = max(np.max(np.abs(arr)), 1e-12)

    # Normalize to [-1, 1]
    norm = np.clip(arr / scale, -1.0, 1.0)

    # Exclusive channels:
    # positive -> green
    # negative -> red
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    rgb[..., 1] = (np.clip(norm, 0, 1) * 255).astype(np.uint8)      # G
    rgb[..., 0] = (np.clip(-norm, 0, 1) * 255).astype(np.uint8)     # R

    matrix_img = Image.fromarray(rgb, mode="RGB")

    if cell_size > 1:
        matrix_img = matrix_img.resize(
            (w * cell_size, h * cell_size),
            resample=Image.Resampling.NEAREST
        )

    canvas_w = matrix_img.width + 2 * margin
    canvas_h = header_height + matrix_img.height + 2 * margin

    canvas = Image.new("RGB", (canvas_w, canvas_h), background)
    draw = ImageDraw.Draw(canvas)

    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 18)
        small_font = ImageFont.truetype("DejaVuSans.ttf", 14)
    except Exception:
        font = ImageFont.load_default()
        small_font = ImageFont.load_default()

    # draw.text((margin, margin), f"Mean: {mean_val:.6g}", fill=text_color, font=font)
    # draw.text((margin, margin + 24), f"Variance: {var_val:.6g}", fill=text_color, font=font)
    # draw.text((margin, margin + 48), f"Shape: {tuple(tensor.shape)} -> shown as {tuple(t.shape)}",
            #   fill=(180, 180, 180), font=small_font)
    print(f"Mean: {mean_val:.6g}")
    print(f"Variance: {var_val:.6g}")
    print(f"Shape: {tuple(tensor.shape)} -> shown as {tuple(t.shape)}")

    canvas.paste(matrix_img, (margin, header_height))

    if out_path is not None:
        canvas.save(out_path)

    return canvas


In [58]:
import torch

# Example tensor
w = torch.randn(25,15) 

img = tensor_weights_to_image(w, out_path="weights.png", cell_size=9)
img.show()

print(w)


Mean: -0.0436901
Variance: 1.19851
Shape: (25, 15) -> shown as (25, 15)
tensor([[-5.9786e-01, -1.6012e+00,  3.8874e-01, -2.8960e-01, -1.5870e+00,
         -1.0975e+00,  1.0174e+00, -8.0572e-02,  1.0335e+00, -1.0923e+00,
         -1.0028e+00, -5.3372e-01,  7.9177e-01,  2.9466e-01,  2.5702e+00],
        [ 1.2880e+00, -1.1762e+00,  6.1182e-01,  2.3122e+00,  1.3977e+00,
          9.4222e-02, -9.5579e-02,  3.2997e-01,  2.2791e-01, -1.1160e-01,
          4.9432e-01, -2.6560e+00,  9.1550e-02, -3.7074e-01,  5.7169e-01],
        [-4.3117e-01,  5.8499e-01, -6.0652e-01, -1.9085e+00, -6.9894e-01,
          2.8352e+00, -1.9573e+00,  5.4557e-01, -1.5206e+00,  2.9169e-01,
         -8.4410e-01, -9.1997e-01,  1.8713e+00, -1.4996e-01,  6.8530e-01],
        [ 1.0513e-01,  4.4780e-01, -1.3803e+00, -1.6168e+00,  7.6640e-01,
          1.8442e+00, -1.5154e+00, -4.3799e-01,  1.2963e+00, -2.4253e-01,
          2.9375e+00, -7.0560e-01,  4.7748e-01,  2.7336e-01,  7.6973e-01],
        [-1.8758e+00, -3.1837e-01, -

In [55]:
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont


def activations_to_grayscale_image(
    activations: torch.Tensor,
    out_path: str | None = None,
    cell_size: int = 4,
    header_height: int = 0,
    margin: int = 0,
    # Normalization controls (pick one style)
    normalize: str = "minmax",   # "minmax" | "symmetric" | "clamp01"
    vmin: float | None = None,
    vmax: float | None = None,
    # Optional contrast shaping
    gamma: float = 1.0,          # >1 darker mids, <1 brighter mids
    background=(0, 0, 0),
    text_color=(240, 240, 240),
) -> Image.Image:
    """
    Visualize neuron activations as grayscale:
      - black = inactive (low)
      - white = active (high)
      - gray tones in between

    Shape handling:
      - scalar -> (1,1)
      - 1D -> (1, N)
      - 2D -> (H, W)
      - >2D -> reshapes to (-1, last_dim)

    Args:
        activations: torch tensor of activations
        out_path: optional save path
        cell_size: pixel scaling per element (NEAREST)
        header_height: reserved space for stats
        margin: padding
        normalize:
          * "minmax": map [min,max] -> [0,1]
          * "symmetric": map [-S,S] -> [0,1] with 0 at mid-gray (useful if activations have negatives)
          * "clamp01": assume already in [0,1], just clamp
        vmin/vmax: override min/max for "minmax" (or clamp range)
        gamma: gamma correction applied after normalization
    """
    if not isinstance(activations, torch.Tensor):
        raise TypeError("activations must be a torch.Tensor")

    t = activations.detach().float().cpu()
    if t.numel() == 0:
        raise ValueError("activations must not be empty")

    # reshape to 2D
    if t.ndim == 0:
        t = t.reshape(1, 1)
    elif t.ndim == 1:
        t = t.unsqueeze(0)
    elif t.ndim > 2:
        t = t.reshape(-1, t.shape[-1])

    h, w = t.shape
    arr = t.numpy()

    mean_val = float(arr.mean())
    var_val = float(arr.var())

    # normalize to [0,1]
    eps = 1e-12
    if normalize == "clamp01":
        a01 = np.clip(arr, 0.0, 1.0)

    elif normalize == "minmax":
        lo = float(np.min(arr)) if vmin is None else float(vmin)
        hi = float(np.max(arr)) if vmax is None else float(vmax)
        denom = max(hi - lo, eps)
        a01 = np.clip((arr - lo) / denom, 0.0, 1.0)

    elif normalize == "symmetric":
        # Map [-S, S] -> [0,1], 0 maps to 0.5 gray
        if vmin is not None or vmax is not None:
            # If user supplied, infer symmetric S from the larger magnitude
            lo = float(np.min(arr)) if vmin is None else float(vmin)
            hi = float(np.max(arr)) if vmax is None else float(vmax)
            S = max(abs(lo), abs(hi), eps)
        else:
            S = max(float(np.max(np.abs(arr))), eps)

        a01 = np.clip((arr / S + 1.0) * 0.5, 0.0, 1.0)

    else:
        raise ValueError("normalize must be one of: 'minmax', 'symmetric', 'clamp01'")

    # gamma correction
    if gamma != 1.0:
        a01 = np.clip(a01, 0.0, 1.0) ** (1.0 / float(gamma))

    gray = (a01 * 255.0).round().astype(np.uint8)
    matrix_img = Image.fromarray(gray, mode="L").convert("RGB")

    if cell_size > 1:
        matrix_img = matrix_img.resize(
            (w * cell_size, h * cell_size),
            resample=Image.Resampling.NEAREST
        )

    canvas_w = matrix_img.width + 2 * margin
    canvas_h = header_height + matrix_img.height + 2 * margin
    canvas = Image.new("RGB", (canvas_w, canvas_h), background)
    draw = ImageDraw.Draw(canvas)

    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 18)
        small_font = ImageFont.truetype("DejaVuSans.ttf", 14)
    except Exception:
        font = ImageFont.load_default()
        small_font = ImageFont.load_default()

    # draw.text((margin, margin), f"Mean: {mean_val:.6g}", fill=text_color, font=font)
    # draw.text((margin, margin + 24), f"Variance: {var_val:.6g}", fill=text_color, font=font)
    # draw.text(
    #     (margin, margin + 48),
    #     f"Shape: {tuple(activations.shape)} -> shown as {tuple(t.shape)} | normalize={normalize}",
    #     fill=(180, 180, 180),
    #     font=small_font
    # )
    print( f"Mean: {mean_val:.6g}")
    print(f"Variance: {var_val:.6g}")
    print(f"Shape: {tuple(activations.shape)} -> shown as {tuple(t.shape)} | normalize={normalize}")

    canvas.paste(matrix_img, (margin, header_height))

    if out_path is not None:
        canvas.save(out_path)

    return canvas


In [56]:
a = torch.relu(torch.randn(20, 1))
img = activations_to_grayscale_image(a, out_path="acts.png", cell_size=9, normalize="minmax")
img.show()


Mean: 0.667782
Variance: 0.53723
Shape: (20, 1) -> shown as (20, 1) | normalize=minmax
